In [0]:
from pyspark.sql.functions import *

In [0]:
tables = {
    "sql_customers" : "customers",
    "sql_products" : "products",
    "sql_sales_transactions" : "sales_transactions"
}
catalog = novamart
schema = bronze
base_path = f"/Volumes/{catalog}/{schema}/landing/landing_zone/sales_db/"
check_point = '/Volumes/novamart/bronze/checkpoint/'
for table_name, folder in tables.items():

    source_path = f"{base_path}/{folder}"
    checkpoint_path = f"{check_point}/{table_name}_bronze"
    df_raw = (spark.readStream
                    .format("cloudFiles")
                    .option("cloudFiles.format", "csv")
                    .option("header", "true")
                    .option("cloudFiles.inferSchema", "true")
                    .option("cloudFiles.schemaLocation", f"{checkpoint_path}/schema")
                    .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
                    .load(source_path))

    df_bronze = (df_raw
                 .withColumns({
                     "ingesttime" , current_timestamp(),
                     "file_name", input_file_name()
                     })
                 )

    (df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{check_point}/write")
        .option("delta.autoOptimze.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .toTable(f"{catalog}.{schema}.{table_name}"))

In [0]:
source_path = f"/Volumes/{catalog}/{schema}/landing/landing_zone/crm_api/customers"
check_point = f"/Volumes/{catalog}/{schema}/checkpoint/crm_customers"
df_raw = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.inferSchema", "true")
          .option("cloudFiles.schemaLocation", f"{check_point}/schema")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(source_path))

df_bronze = (df_raw
             .withColumn("ingesttime", current_timestamp())
             .withColumn("file_name", input_file_name()))

(df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{check_point}/write")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .toTable(f"{catalog}.{schema}.crm_customers"))

In [0]:
source_path_clicksteam = f"/Volumes/{catalog}/{schema}/landing/landing_zone/kafka"
check_point_clickstream = f"/Volumes/{catalog}/{schema}/checkpoint/kafka_clcikstream"

df_raw = (spark.readStream
          .format("cloudFiles")
          .option("cloudFiles.format", "json")
          .option("cloudFiles.inferSchema", "true")
          .option("cloudFiles.schemaLocation", f"{check_point_clickstream}/schema")
          .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
          .load(source_path_clicksteam))

df_bronze = (df_raw
             .withColumn("ingesttime", current_timestamp())
             .withColumn("file_name", input_file_name()))

(df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", f"{check_point_clickstream}/write")
        .option("delta.autoOptimize.optimizeWrite", "true")
        .option("delta.autoOptimize.autoCompact", "true")
        .toTable(f"{catalog}.{schema}.clickstream"))